# ConvTran Reproduction: UEA Benchmark

After testing the pipeline on BasicMotions, we extend the evaluation to the 30 UEA datasets used in the paper.

This notebook records data preparation, experiment settings, results and problems encountered during reproduction.

Each dataset and training seed is run in a separate process using `scripts/run_baseline.py`. Results are collected in `results/baseline_runs.csv`.

BasicMotions pilot results are documented in `01_reproduction.ipynb`.

In [15]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

ROOT = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "upstream" / "ConvTran" / "main.py").is_file()
    ),
    None,
)

assert ROOT is not None, "Open the notebook from the project repository."

DATA_ROOT = ROOT / "Dataset" / "UEA" / "Multivariate_ts"
RUNNER = ROOT / "scripts" / "run_baseline.py"
RESULTS_PATH = ROOT / "results" / "baseline_runs.csv"

print("Repository:", ROOT)
print("Kernel Python:", sys.executable)

Repository: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction
Kernel Python: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/bin/python


## Experiment settings

We begin with the same released-code baseline used in the pilot:

- 100 epochs.
- Batch size of 16.
- Training seeds: 1234, 2345, 3456, 4567 and 5678.
- Checkpoint selected by the lowest validation loss.
- Official UEA train/test splits.
- Fixed training/validation split from the upstream loader.

These settings follow the implementation defaults rather than an exactly verified configuration for the paper's reported results.

Dataset-specific changes, if needed, will be recorded separately.

In [16]:
UEA_DATASETS = [
    "ArticularyWordRecognition",
    "AtrialFibrillation",
    "BasicMotions",
    "CharacterTrajectories",
    "Cricket",
    "DuckDuckGeese",
    "EigenWorms",
    "Epilepsy",
    "ERing",
    "EthanolConcentration",
    "FaceDetection",
    "FingerMovements",
    "HandMovementDirection",
    "Handwriting",
    "Heartbeat",
    "InsectWingbeat",
    "JapaneseVowels",
    "Libras",
    "LSST",
    "MotorImagery",
    "NATOPS",
    "PenDigits",
    "PEMS-SF",
    "PhonemeSpectra",
    "RacketSports",
    "SelfRegulationSCP1",
    "SelfRegulationSCP2",
    "SpokenArabicDigits",
    "StandWalkJump",
    "UWaveGestureLibrary",
]

assert len(UEA_DATASETS) == 30

## Downloading the UEA archive

We download the official archive and extract the TRAIN and TEST files for each dataset. Existing files are preserved.

In [17]:
import shutil
import zipfile

import requests
from tqdm.auto import tqdm

archive_url = (
    "https://www.timeseriesclassification.com/"
    "aeon-toolkit/Archives/Multivariate2018_ts.zip"
)

archive_dir = ROOT / "Dataset" / "archives"
archive_dir.mkdir(parents=True, exist_ok=True)
archive_path = archive_dir / "Multivariate2018_ts.zip"

if not archive_path.exists():
    partial_path = archive_path.with_suffix(".zip.part")

    with requests.get(
        archive_url, stream=True, timeout=(30, 120)
    ) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))

        with (
            partial_path.open("wb") as output,
            tqdm(
                total=total or None,
                unit="B",
                unit_scale=True,
                desc="Downloading UEA",
            ) as progress,
        ):
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
                    progress.update(len(chunk))

    if not zipfile.is_zipfile(partial_path):
        raise ValueError("The downloaded file is not a valid ZIP archive.")

    partial_path.replace(archive_path)
else:
    print("Using existing archive:", archive_path)

with zipfile.ZipFile(archive_path) as archive:
    members = {}

    for name in UEA_DATASETS:
        for split in ("TRAIN", "TEST"):
            filename = f"{name}_{split}.ts"
            matches = [
                member
                for member in archive.namelist()
                if Path(member).name == filename
            ]
            if len(matches) != 1:
                raise ValueError(
                    f"Expected one archive member for {filename}, "
                    f"found {len(matches)}."
                )
            members[(name, filename)] = matches[0]

    for (name, filename), member in tqdm(
        members.items(), desc="Extracting split files"
    ):
        destination = DATA_ROOT / name / filename
        if destination.exists():
            continue

        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_suffix(".ts.part")

        with archive.open(member) as source, partial.open("wb") as target:
            shutil.copyfileobj(source, target)

        partial.replace(destination)

print("Dataset extraction completed.")

Using existing archive: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/Dataset/archives/Multivariate2018_ts.zip


Extracting split files: 100%|██████████| 60/60 [00:00<00:00, 130460.47it/s]

Dataset extraction completed.


## Data availability

We check that both official split files are present for each dataset.
File presence alone does not confirm that the files are complete or that a dataset can be trained with the available memory.

Missing datasets must be downloaded before training.

In [18]:
inventory = pd.DataFrame([
    {
        "dataset": name,
        "train_file": (
            DATA_ROOT / name / f"{name}_TRAIN.ts"
        ).is_file(),
        "test_file": (
            DATA_ROOT / name / f"{name}_TEST.ts"
        ).is_file(),
    }
    for name in UEA_DATASETS
])

inventory["files_present"] = (
    inventory["train_file"] & inventory["test_file"]
)

print(
    f"Datasets with both files: "
    f"{inventory['files_present'].sum()}/{len(inventory)}"
)

inventory

Datasets with both files: 30/30


,dataset,train_file,test_file,files_present
0,ArticularyWordRecognition,True,True,True
1,AtrialFibrillation,True,True,True
2,BasicMotions,True,True,True
3,CharacterTrajectories,True,True,True
4,Cricket,True,True,True
5,DuckDuckGeese,True,True,True
6,EigenWorms,True,True,True
7,Epilepsy,True,True,True
8,ERing,True,True,True
9,EthanolConcentration,True,True,True


## Training

Training is managed by `scripts/run_baseline.py`.

For example, from the project root:

    uv run python scripts/run_baseline.py --dataset BasicMotions

The script reuses completed runs and updates the results CSV.
Training commands for the remaining datasets will be added after checking their data and memory requirements.

## Results collected so far

The following table loads completed runs from the results CSV.
It includes the BasicMotions runs collected during the pilot.

In [19]:
assert RESULTS_PATH.is_file(), "No results CSV found."

runs = pd.read_csv(RESULTS_PATH)

required_columns = {
    "dataset",
    "seed",
    "epochs",
    "batch_size",
    "accuracy",
    "test_loss",
}

assert required_columns.issubset(runs.columns), (
    "The results CSV is missing required columns."
)
assert not runs.duplicated(["dataset", "seed"]).any(), (
    "Duplicate dataset/seed entries found."
)
assert runs["accuracy"].between(0, 1).all(), (
    "Accuracy values must be between 0 and 1."
)

runs = runs[runs["dataset"].isin(UEA_DATASETS)].copy()
runs.sort_values(["dataset", "seed"])

,dataset,seed,epochs,batch_size,accuracy,test_loss
0,BasicMotions,1234,100,16,1.0,0.161723
1,BasicMotions,2345,100,16,1.0,0.183982
2,BasicMotions,3456,100,16,1.0,0.210695
3,BasicMotions,4567,100,16,1.0,0.159011
4,BasicMotions,5678,100,16,1.0,0.182155


In [20]:
EXPECTED_SEEDS = {1234, 2345, 3456, 4567, 5678}

summary = runs.groupby("dataset").agg(
    completed_runs=("seed", "count"),
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
)

seed_sets = runs.groupby("dataset")["seed"].agg(set)
summary["all_seeds_present"] = seed_sets.apply(
    lambda seeds: seeds == EXPECTED_SEEDS
)

progress = (
    inventory[["dataset", "files_present"]]
    .merge(summary.reset_index(), on="dataset", how="left")
)

progress["completed_runs"] = (
    progress["completed_runs"].fillna(0).astype(int)
)
progress["all_seeds_present"] = (
    progress["all_seeds_present"].fillna(False).astype(bool)
)
progress["accuracy_mean_pct"] = progress["accuracy_mean"] * 100
progress["accuracy_std_pp"] = progress["accuracy_std"] * 100

progress[
    [
        "dataset",
        "files_present",
        "completed_runs",
        "all_seeds_present",
        "accuracy_mean_pct",
        "accuracy_std_pp",
    ]
].round(4)

,dataset,files_present,completed_runs,all_seeds_present,accuracy_mean_pct,accuracy_std_pp
0,ArticularyWordRecognition,True,0,False,NaN,NaN
1,AtrialFibrillation,True,0,False,NaN,NaN
2,BasicMotions,True,5,True,100.0,0.0
3,CharacterTrajectories,True,0,False,NaN,NaN
4,Cricket,True,0,False,NaN,NaN
5,DuckDuckGeese,True,0,False,NaN,NaN
6,EigenWorms,True,0,False,NaN,NaN
7,Epilepsy,True,0,False,NaN,NaN
8,ERing,True,0,False,NaN,NaN
9,EthanolConcentration,True,0,False,NaN,NaN


## Comparison with the paper

The comparison table will be added once the reported values and
their evaluation protocol have been checked.

We will report our mean accuracy and standard deviation across seeds,
alongside the paper's values. Differences in model configuration or
evaluation procedure will be stated explicitly.

## Experiment notes

BasicMotions has five completed runs from the pilot, each with
100% test accuracy.

The remaining datasets have not yet been evaluated.
